# DiffBIR: שחזור גלובלי של 5 תמונות הניסוי

מריצים את **DiffBIR** (`XPixelGroup/DiffBIR`, רישיון Apache 2.0) על 5 התמונות במצב שחזור כללי (`--task sr`) **בלי הגדלה** (`--upscale 1`), כך שהפלט באותו גודל כמו המקור. הפלט הגולמי מורידים כ-zip. החזרת הפנים מהמקור ובדיקת ההבדלים מול המקור נעשות **מקומית** (`restore.py --engine external --raw ...`), לא ב-Colab.

**לפני שמתחילים**
1. `Runtime ← Change runtime type ← T4 GPU`, **ועם High-RAM** (בלי זה התהליך נהרג מחוסר זיכרון: `exit=-9` בזמן טעינת המשקלים).
2. מריצים את התאים לפי הסדר.
3. התמונות עולות ל-VM של Colab. מעלים רק את 5 תמונות הניסוי (יש עליהן הרשאה), והתא האחרון מוחק אותן.
4. הורץ בפועל ב-21.10.2026 על T4 High-RAM עם Python 3.13: ה-`requirements.txt` של DiffBIR נועל `torch==2.2.2`, שלא קיים שם, ולכן התא מתקין את התלויות בלי הנעילות, ומתקן ב-`sed` את `torch.Tuple` שהוסר מ-PyTorch. v2.1 על 5 התמונות לקח כ-23 דקות (בלי הורדת המשקלים, 5GB, כ-3 דקות).
5. התמונות עולות ל-VM של Colab. מעלים רק את 5 תמונות הניסוי (יש עליהן הרשאה), והתא האחרון מוחק אותן. בסיום: `Runtime ← Disconnect and delete runtime`.

**שני וריאנטים:** `v2.1` (`cfg_scale 8`) ו-`v2` (`cfg_scale 4`). שניהם בלי captioner (`--captioner none`), כדי לחסוך זיכרון ולא להוריד מודל שפה. משקלי DiffBIR יורדים אוטומטית בהרצה הראשונה.

## 0. הכנה ובדיקת GPU

In [ ]:
import os, sys, time, shutil, subprocess, zipfile
from pathlib import Path
from PIL import Image

ROOT = Path("/content")
INPUTS = ROOT / "inputs_diffbir"
OUT = ROOT / "output_diffbir"
LOG = OUT / "log.csv"
EXPECTED = {"1_cracks", "2_stains", "3_tear", "4_blur", "5_faces"}

INPUTS.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
if not LOG.exists():
    print("variant,seconds,ok,note", file=open(LOG, "w", encoding="utf-8"))

In [ ]:
import torch

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "|", gpu)
if not torch.cuda.is_available():
    raise SystemExit("Switch the runtime to a GPU first (Runtime > Change runtime type).")

## 1. העלאת 5 התמונות מהמחשב
בוחרים את הקבצים `1_cracks`, `2_stains`, `3_tear`, `4_blur`, `5_faces` (המקוריים, **לא** העותקים המסומנים באדום).

In [ ]:
from google.colab import files

uploaded = files.upload()
for name, data in uploaded.items():
    (INPUTS / name).write_bytes(data)

stems = {p.stem for p in INPUTS.iterdir()}
print("uploaded:", sorted(stems))
if stems != EXPECTED:
    print("WARNING - expected exactly:", sorted(EXPECTED))

## 2. התקנת DiffBIR
התא משכפל את הריפו, מתקין את התלויות (בלי הנעילות הישנות של `torch`, `xformers` ושאר החבילות הכבדות שאין בהן צורך עם `--captioner none`), ורושם את ה-commit ל-`versions.txt`. אם `pip` מתקין גרסת PyTorch אחרת, ייתכן ש-Colab יבקש הפעלה מחדש של ה-runtime: מפעילים מחדש ומריצים שוב מהתא של ההכנה (ההעלאה נשמרת ב-`/content`).

In [ ]:
%cd /content
!git clone -q https://github.com/XPixelGroup/DiffBIR.git
%cd /content/DiffBIR
!pip install -q omegaconf accelerate einops opencv-python scipy ftfy regex timm pytorch-lightning lpips torchsde
!pip install -q --no-deps facexlib
# the repo uses typing aliases that newer PyTorch removed
!grep -rl torch.Tuple diffbir | xargs -r sed -i s/torch.Tuple/tuple/g
!grep -rl torch.List diffbir | xargs -r sed -i s/torch.List/list/g
!grep -rl torch.Dict diffbir | xargs -r sed -i s/torch.Dict/dict/g

sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd="/content/DiffBIR", capture_output=True, text=True).stdout.strip()
with open(OUT / "versions.txt", "a", encoding="utf-8") as f:
    print(f"DiffBIR {sha}; torch {torch.__version__}; gpu {gpu}", file=f)
print("DiffBIR", sha)

## 3. הרצה
פונקציה אחת שמריצה וריאנט ורושמת זמן ותוצאה. הדגלים של טילינג (`--cleaner_tiled` ושלושת האחרים) מפחיתים שימוש בזיכרון ב-T4. אם עדיין יש `CUDA out of memory`, מקטינים את `--cldm_tile_size` ל-384 או 256.

In [ ]:
TILING = ["--cleaner_tiled", "--cleaner_tile_size", "256", "--cleaner_tile_stride", "128",
          "--vae_encoder_tiled", "--vae_encoder_tile_size", "256",
          "--vae_decoder_tiled", "--vae_decoder_tile_size", "256",
          "--cldm_tiled", "--cldm_tile_size", "512", "--cldm_tile_stride", "256"]
NEG = "low quality, blurry, low-resolution, noisy, unsharp, weird textures"


def run_variant(name, extra):
    out = OUT / name
    out.mkdir(exist_ok=True)
    cmd = ["python", "-u", "inference.py", "--task", "sr", "--upscale", "1",
           "--sampler", "spaced", "--steps", "50", "--captioner", "none",
           "--pos_prompt", "", "--neg_prompt", NEG,
           "--input", str(INPUTS), "--output", str(out), "--device", "cuda", "--precision", "fp32",
           *TILING, *extra]
    t = time.time()
    p = subprocess.run(cmd, cwd="/content/DiffBIR", capture_output=True, text=True)
    seconds = time.time() - t
    made = sorted(q.name for q in out.rglob("*") if q.suffix.lower() in {".png", ".jpg", ".jpeg"})
    ok = p.returncode == 0 and len(made) >= len(EXPECTED)
    print(f"[{name}] exit={p.returncode} {seconds:.0f}s outputs={len(made)}")
    if not ok:
        print(p.stdout[-2000:])
        print(p.stderr[-3000:])
    with open(LOG, "a", encoding="utf-8") as f:
        print(f"{name},{seconds:.1f},{ok},{len(made)} files", file=f)
    return ok

In [ ]:
# v2.1: newer model, cfg_scale 8
run_variant("diffbir_v21", ["--version", "v2.1", "--cfg_scale", "8", "--noise_aug", "0"])

In [ ]:
# v2: the ECCV version, cfg_scale 4
run_variant("diffbir_v2", ["--version", "v2", "--cfg_scale", "4"])

## 4. בדיקת גדלים והורדה
הפלט אמור להיות באותו גודל כמו הקלט (`--upscale 1`). אם לא, ההתאמה לגודל המקור תיעשה מקומית. אחרי ההורדה מריצים מקומית לכל תמונה:

`python restore.py experiments/images/<שם>.jpg --engine external --raw <הפלט> --storage storage/diffbir`

In [ ]:
for variant_dir in sorted(OUT.iterdir()):
    if not variant_dir.is_dir():
        continue
    for img in sorted(variant_dir.rglob("*")):
        if img.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
            continue
        src = next((p for p in INPUTS.iterdir() if p.stem == img.stem), None)
        a = Image.open(img).size
        b = Image.open(src).size if src else None
        print(f"{variant_dir.name}/{img.name}: output {a} input {b}", "" if a == b else "  <-- size differs")

zip_path = ROOT / "output_diffbir.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(OUT))
files.download(str(zip_path))

## 5. ניקוי
מוחק את התמונות שהועלו ואת הפלטים מה-VM.

In [ ]:
shutil.rmtree(INPUTS, ignore_errors=True)
shutil.rmtree(OUT, ignore_errors=True)
(ROOT / "output_diffbir.zip").unlink(missing_ok=True)
print("cleaned")